In [31]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import re
import pickle
from collections import defaultdict

In [3]:
df1 = pd.read_csv("../data/expenses_2023_2024_2025.csv")
df2 = pd.read_csv("../data/expenses_2026.csv")
len(df1) , len(df2)

(724, 65)

In [ ]:
df = pd.concat([df1, df2], ignore_index = True)
df

In [ ]:
df.drop(columns = ["Date","Amount","Receipt","Created","Notes"], inplace = True)
df

In [6]:
df[df.isna().any(axis=1)]


,Name,Category
55,New Expense,NaN
138,New Expense,NaN
311,New Expense,NaN
788,New Expense,NaN


In [7]:
df.dropna(inplace = True)
df[df.isna().any(axis=1)]

,Name,Category


In [8]:
df["Category_clean"] = df["Category"].str.replace(r" .*",'',regex = True)

df["Category_clean"]

0      Education
1      Utilities
2      Transport
3           Food
4      Transport
         ...    
783         Food
784         Food
785         Rent
786         Food
787    Groceries
Name: Category_clean, Length: 785, dtype: str

In [9]:
df["Category_clean"].unique()

<StringArray>
[     'Education',      'Utilities',      'Transport',           'Food',
       'Shopping',        'Charity',      'Groceries',           'Home',
           'Rent',         'Travel',        'Housing',         'Health',
 'Entertainments',  'subscriptions']
Length: 14, dtype: str

In [10]:
df.drop(columns = "Category",inplace = True)

In [ ]:
def clean_merchant(name: str) -> str:
    if not isinstance(name, str):
        return ""
    name = name.lower().strip()                  # lowercase + strip whitespace
    name = re.sub(r"[^a-z0-9\s]", "", name)     # remove special characters
    name = re.sub(r"\s+", " ", name)             # collapse extra whitespace
    return name

df["Name_clean"] = df["Name"].apply(clean_merchant)
df[df["Name"] != df["Name_clean"]][["Name", "Name_clean"]].head(20)


In [ ]:

print(f"Before: {df['Name'].nunique()} unique merchants")
print(f"After:  {df['Name_clean'].nunique()} unique merchants")

# Show groups that got merged
grouped = df.groupby("Name_clean")["Name"].unique()
merged = grouped[grouped.apply(len) > 1]
print("\nMerged duplicates:")
print(merged)

In [13]:
df.drop(columns = "Name",inplace = True)

In [14]:
df.head(5)

,Category_clean,Name_clean
0,Education,pte
1,Utilities,aws
2,Transport,lyft
3,Food,mima cake smol
4,Transport,lyft


In [15]:
# improvements need to be made to the model
# need to clean the data better prob
model = SentenceTransformer("all-MiniLM-L6-v2")
df["embeddings"] = model.encode(df["Name_clean"].tolist()).tolist()
df.to_csv("../data/expenses_embeddings.csv")

with open("embedding_model.pkl", 'wb') as file:
    pickle.dump(model,file)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9613.54it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Testing

In [16]:
print(df["embeddings"].iloc[0])
print(type(df["embeddings"].iloc[0]))

[-0.09707318991422653, -0.01636601984500885, 0.06500667333602905, -0.005695380736142397, -0.0672595426440239, 0.0015950577799230814, 0.1461736112833023, 0.1296587437391281, -0.009487017057836056, 0.023672934621572495, -0.03715457022190094, -0.01877420023083687, -0.052396878600120544, 0.035756368190050125, 0.0009263672982342541, -0.0713868960738182, -0.0014089297037571669, -0.05849980562925339, 0.033942922949790955, -0.05514450743794441, -0.03897974267601967, 0.018957166001200676, -0.035959433764219284, 0.015516026876866817, -0.04866412654519081, 0.006682234350591898, -0.0022971995640546083, -0.003111674217507243, -0.031232498586177826, -0.05906848981976509, 0.010555167682468891, 0.01231620367616415, -0.060420773923397064, -0.013408325612545013, 0.016133207827806473, -0.0048719667829573154, 0.023413248360157013, -0.016886873170733452, 0.002434094436466694, -0.015014396980404854, 0.0577872097492218, -0.07372475415468216, -0.056742891669273376, -0.04538207873702049, 0.02156810089945793, 0

In [17]:
stored_embeddings = np.array(df["embeddings"].tolist())
stored_embeddings

array([[-0.09707319, -0.01636602,  0.06500667, ..., -0.00585678,
         0.05942262,  0.01154883],
       [ 0.00534156,  0.03216606, -0.02018812, ...,  0.00733654,
        -0.03691204,  0.0270026 ],
       [-0.02770441, -0.04861553,  0.0348378 , ...,  0.11941555,
         0.11628473, -0.04487879],
       ...,
       [ 0.0345207 ,  0.01578971,  0.00928153, ..., -0.01730966,
        -0.01378625, -0.05329339],
       [ 0.02888749,  0.01562454,  0.02737226, ...,  0.05943118,
         0.10111742, -0.04254457],
       [-0.01583275,  0.00145389, -0.01213112, ..., -0.0189964 ,
         0.05268766,  0.04807347]], shape=(785, 384))

In [18]:
def cosine_similarity(a, b):
    return np.dot(a, b.T) / (np.linalg.norm(a) * np.linalg.norm(b, axis=1))

In [19]:
def categorize(merchant_name: str, threshold: float = 0.80):
    query_embedding = model.encode([merchant_name])[0]
    #print("query = ",query_embedding)
    scores = cosine_similarity(query_embedding, stored_embeddings)
    #print("score = ",scores)
    
    top_indices = scores.argsort()[::-1][:3]  # top 3 matches
    
    top_score = scores[top_indices[0]]
    top_category = df.iloc[top_indices[0]]["Category_clean"]
    top_merchant = df.iloc[top_indices[0]]["Name_clean"]

    print(top_indices)
    print(top_score)
    print(top_category)
    print(top_merchant)
    #return
    if top_score >= threshold:
        return {
            "category": top_category,
            "confidence": float(top_score),
            "matched_to": top_merchant,
            "source": "embeddings"
        }
    else:
        top_matches = [
            (df.iloc[i]["Name_clean"], df.iloc[i]["Category_clean"], float(scores[i]))
            for i in top_indices
        ]
        return {
            "category": None,
            "confidence": float(top_score),
            "top_matches": top_matches,
            "source": "needs_llm"
        }

In [71]:
TOP_N = 10

def categorize2(merchant_name: str, bank_category: str = "", threshold: float = 0.60):
    query_embedding = model.encode([merchant_name])[0]
    scores = cosine_similarity(query_embedding, stored_embeddings)
    
    top_indices = scores.argsort()[::-1][:TOP_N]
    top_matches = [
        (df.iloc[i]["Name_clean"], df.iloc[i]["Category_clean"], float(scores[i]))
        for i in top_indices
    ]

    category_scores = defaultdict(float)
    for _, category, score in top_matches:
        category_scores[category] += score

    category_normalized = {
        cat: total / TOP_N
        for cat, total in category_scores.items()
    }

    print("\nCategory scores (normalized):")
    for cat, score in sorted(category_normalized.items(), key=lambda x: -x[1]):
        print(f"  {cat}: {score:.3f}")

    best_category = max(category_normalized, key=category_normalized.get)
    best_score = category_normalized[best_category]

    print(f"\n→ Winner: {best_category} (score={best_score:.3f})")

    if best_score >= threshold:
        source = "embeddings"
        category = best_category
    else:
        source = "needs_llm"
        category = None

    return {
        "category": category,
        "confidence": best_score,
        "top_matches": top_matches,
        "source": source,
    }

In [66]:
from collections import defaultdict
import numpy as np

TOP_N = 10
EXACT_MATCH_THRESHOLD = 0.99

def cosine_similarity(a, b):
    return np.dot(a, b.T) / (np.linalg.norm(a) * np.linalg.norm(b, axis=1))

def categorize3(merchant_name: str, bank_category: str = "", threshold: float = 0.60):
    query_embedding = model.encode([merchant_name])[0]
    scores = cosine_similarity(query_embedding, stored_embeddings)
    
    top_indices = scores.argsort()[::-1][:TOP_N]
    top_matches = [
        (df.iloc[i]["Name_clean"], df.iloc[i]["Category_clean"], float(scores[i]))
        for i in top_indices
    ]

    # Voting with squared scores to amplify high confidence matches
    category_scores = defaultdict(float)
    for _, category, score in top_matches:
        category_scores[category] += score ** 2

    category_normalized = {
        cat: total / TOP_N
        for cat, total in category_scores.items()
    }

    best_category = max(category_normalized, key=category_normalized.get)
    best_score = category_normalized[best_category]

    print("\nCategory scores (normalized):")
    for cat, score in sorted(category_normalized.items(), key=lambda x: -x[1]):
        print(f"  {cat}: {score:.3f}")
    print(f"\n→ Voting winner: {best_category} (score={best_score:.3f})")

    # Early exit — exact match AND voting agrees
    top_name, top_category, top_score = top_matches[0]
    if top_score >= EXACT_MATCH_THRESHOLD and top_category == best_category:
        print(f"✅ Exact match confirmed by voting: {top_name} → {top_category}")
        return {
            "category": top_category,
            "confidence": best_score,
            "top_matches": top_matches,
            "source": "embeddings",
        }

    # Normal voting threshold
    if best_score >= threshold:
        source = "embeddings"
        category = best_category
    else:
        source = "needs_llm"
        category = None

    return {
        "category": category,
        "confidence": best_score,
        "top_matches": top_matches,
        "source": source,
    }

In [41]:
result = categorize("walmart")
result

[413 390 389]
1.0000000222293368
Shopping
walmart


{'category': 'Shopping',
 'confidence': 1.0000000222293368,
 'matched_to': 'walmart',
 'source': 'embeddings'}

In [72]:
result = categorize2("walmart")
result


Category scores (normalized):
  Groceries: 0.800
  Shopping: 0.200

→ Winner: Groceries (score=0.800)


{'category': 'Groceries',
 'confidence': 0.8000000177834694,
 'top_matches': [('walmart', 'Shopping', 1.0000000222293368),
  ('walmart', 'Shopping', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368)],
 'source': 'embeddings'}

In [73]:
result = categorize3("walmart")
result


Category scores (normalized):
  Groceries: 0.800
  Shopping: 0.200

→ Voting winner: Groceries (score=0.800)


{'category': 'Groceries',
 'confidence': 0.8000000355669392,
 'top_matches': [('walmart', 'Shopping', 1.0000000222293368),
  ('walmart', 'Shopping', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368),
  ('walmart', 'Groceries', 1.0000000222293368)],
 'source': 'embeddings'}